In [ ]:
import pandas as pd
import os
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

Cargamos el archivo `corpus.csv` y visualizamos las primeras filas para entender su estructura.

In [9]:
ruta_input = "corpus.csv"

if os.path.exists(ruta_input):
    df = pd.read_csv(ruta_input, encoding='utf-8')
    print(f"¡Éxito! Se cargaron {len(df)} películas.")
    display(df.head(3))
else:
    print(f"Error: No se encontró el archivo '{ruta_input}' en el directorio.")

¡Éxito! Se cargaron 70 películas.


,titulo,idioma,sinopsis,reseña,género
0,Avatar,Español,Exploramos en Avatar la historia de una serie ...,"Tras verla, diría que es un poco lenta, pero c...","Acción, Aventura, Fantasía, Ciencia Ficción"
1,Pirates of the Caribbean: At World's End,Inglés,Acompañamos a los personajes de Pirates of the...,"Desde mi punto de vista, estamos ante impactan...","Aventura, Fantasía, Acción"
2,Spectre,Español,Acompañamos a los personajes de Spectre en un ...,Una experiencia cinematográfica fascinante y m...,"Acción, Aventura, Crimen"


Para cada película, unimos los siguientes campos en un único texto:
- `titulo` (Título de la película)
- `sinopsis` (Sinopsis de la película)
- `reseña` (Reseña de los usuarios)
- `género` (Géneros de la película)

Reemplazamos posibles valores nulos (`NaN`) con cadenas vacías para evitar errores de concatenación.
Además, aplicamos técnicas de normalización usando NLTK (conversión a minúsculas, tokenización, eliminación de signos de puntuación y palabras vacías, y reducción a la raíz o stemming) tal como recomienda el PDF de Recuperación de la Información.

In [10]:
# Rellenar valores nulos con una cadena vacía
df = df.fillna("")

# Configurar para español
stop_words = set(stopwords.words('spanish'))
stemmer = SnowballStemmer('spanish')

def procesar_texto(texto):
    # Convertir a minúsculas
    texto = texto.lower()
    # Tokenizar
    tokens = word_tokenize(texto)
    # Eliminar caracteres no alfanuméricos, eliminar palabras vacías y aplicar stemming
    tokens_procesados = [
        stemmer.stem(re.sub(r'[^\w]', '', token)) 
        for token in tokens 
        if re.search(r'\w', token) and token not in stop_words
    ]
    return " ".join(tokens_procesados)

# Crear una columna con el texto concatenado
df['documento_crudo'] = (
    df['titulo'] + 
    ". " + df['sinopsis'] + 
    " " + df['reseña'] + 
    " " + df['género']
)

# Crear la columna 'documento' procesada para el motor de búsqueda
df['documento'] = df['documento_crudo'].apply(procesar_texto)

In [11]:
print(f"Película: {df['titulo'].iloc[0]}\n")
print("Documento generado para búsqueda:")
print("-" * 80)
print(df['documento'].iloc[0])
print("-" * 80)

Película: Avatar

Documento generado para búsqueda:
--------------------------------------------------------------------------------
avat explor avat histori seri event misteri mantien suspens final ideal amant cin mensaj potent tras verl dir lent actuacion brillant recomend accion aventur fantas cienci ficcion
--------------------------------------------------------------------------------


Guardamos el DataFrame resultante en un nuevo archivo CSV.

In [12]:
ruta_output = "corpus_procesado.csv"
df.to_csv(ruta_output, index=False, encoding='utf-8')
print(f" El corpus procesado se guardó en: {ruta_output}")

 El corpus procesado se guardó en: corpus_procesado.csv


### Creación del Índice Invertido con Whoosh
Utilizamos la biblioteca `Whoosh` para construir un índice invertido. Definiremos un esquema con el título y el contenido procesado, crearemos un directorio para el índice y añadiremos cada documento del corpus.

In [13]:
import os
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID

# Definir el esquema del índice
# Usamos el texto ya procesado (en 'documento'), por lo que el campo TEXT tokenizará por espacios correctamente.
schema = Schema(titulo=TEXT(stored=True), 
                contenido=TEXT(stored=True), 
                id=ID(stored=True, unique=True))

# Crear el directorio para el índice si no existe
index_dir = "indexdir"
if not os.path.exists(index_dir):
    os.mkdir(index_dir)

# Crear el índice
ix = create_in(index_dir, schema)

# Abrir un escritor para añadir documentos
writer = ix.writer()

# Iterar sobre el dataframe para añadir los documentos al índice
for i, row in df.iterrows():
    writer.add_document(
        id=str(i),
        titulo=str(row['titulo']),
        contenido=str(row['documento'])
    )

# Confirmar los cambios
writer.commit()
print("Índice invertido creado con éxito en el directorio 'indexdir'.")

Índice invertido creado con éxito en el directorio 'indexdir'.
